### Features and chronological splits

This notebook focuses on TRANSFER and CASH_OUT transactions.

I split the first 400 simulation hours into training, validation and test
sets, targeting roughly 70%, 15% and 15% of transactions. Each hour stays
entirely within one split.

Steps after 400 form a separate stress set because transaction volume
and fraud prevalence change substantially in that period.

The features come from src/features.py. Model A includes the account-draining
indicator and amount-to-balance ratio. Model B excludes those two features
to examine how much performance depends on them, while retaining raw balances.

In [1]:
import sys
sys.path.append("..")

import os
import pandas as pd
from src.features import build_features, MODEL_B_DROP

df = pd.read_parquet("../data/paysim.parquet")
tc = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(tc.shape) 

(2770409, 11)


In [2]:
main = tc[tc["step"] <= 400]
stress = tc[tc["step"] > 400]
print(len(main), len(stress))

2529536 240873


In [3]:
# Cumulative share of main-period rows, step by step
share = main.groupby("step").size().cumsum() / len(main)

# Last step that keeps us at or under 70% and 85% of rows
train_end = share[share <= 0.70].index.max()
val_end = share[share <= 0.85].index.max()
print("train_end:", train_end, "| val_end:", val_end)

train = main[main["step"] <= train_end]
val = main[(main["step"] > train_end) & (main["step"] <= val_end)]
test = main[main["step"] > val_end]

train_end: 299 | val_end: 349


In [4]:
def summarise(name, d):
    return {
        "set": name,
        "rows": len(d),
        "frauds": int(d["isFraud"].sum()),
        "fraud_rate_%": round(d["isFraud"].mean() * 100, 3),
        "first_step": int(d["step"].min()),
        "last_step": int(d["step"].max()),
    }

summary = pd.DataFrame([
    summarise("train", train),
    summarise("val", val),
    summarise("test", test),
    summarise("stress", stress),
])
summary

,set,rows,frauds,fraud_rate_%,first_step,last_step
0,train,1766373,3393,0.192,1,299
1,val,381950,494,0.129,300,349
2,test,381213,590,0.155,350,400
3,stress,240873,3736,1.551,401,743


In [5]:
X = build_features(train.head(1000))
print(X.columns.tolist())
X.describe().T

['is_transfer', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'orig_balance_zero', 'dest_balance_zero', 'drains_account', 'amount_to_balance']


,count,mean,std,min,25%,50%,75%,max
is_transfer,1000.0,4.100000e-01,4.920794e-01,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00
amount,1000.0,2.929862e+05,5.966080e+05,181.000000,62945.817500,144812.000000,3.022797e+05,1.000000e+07
oldbalanceOrg,1000.0,1.284623e+05,7.525699e+05,0.000000,0.000000,0.000000,2.002628e+04,1.293042e+07
oldbalanceDest,1000.0,1.645293e+06,2.994477e+06,0.000000,81572.685000,396274.550000,1.503062e+06,1.927463e+07
orig_balance_zero,1000.0,6.090000e-01,4.882186e-01,0.000000,0.000000,1.000000,1.000000e+00,1.000000e+00
dest_balance_zero,1000.0,6.700000e-02,2.501471e-01,0.000000,0.000000,0.000000,0.000000e+00,1.000000e+00
drains_account,1000.0,2.900000e-02,1.678904e-01,0.000000,0.000000,0.000000,0.000000e+00,1.000000e+00
amount_to_balance,391.0,1.488001e+02,1.203448e+03,0.004139,0.837254,2.746669,1.212925e+01,2.176103e+04


In [6]:
os.makedirs("../data/processed", exist_ok=True)
for name, d in [("train", train), ("val", val), ("test", test), ("stress", stress)]:
    d.to_parquet(f"../data/processed/{name}.parquet")
print("saved")

saved
